# H2-Speicher: ESL / ASL aus PyPSA-DE (Phase 0/1)

Auswertung der gelösten Netze je Stützjahr:
- Phase-0-Checks: Zeitauflösung, Kavernen vs. Tanks, Bilanzschluss
- Phase-1-Kennzahlen: ASL, ESL, ESL/ASL, Volllaststunden, AGV, Zyklen
- Energiebilanz der deutschen H2-Busse (Zerlegung von R(t)), Dauerlinien, Spitzenzeitpunkte

Einheiten: Modell in Heizwert (Hi). NEP in Brennwert: Hs = 1,189 × Hi.
Vorzeichen Speicher: > 0 = Ausspeicherung (ASL), < 0 = Einspeicherung (ESL).
Die Rechenlogik liegt in `h2_storage_check.py` im selben Ordner.

In [0]:
%pip install -q pypsa

In [0]:
# ==================== PRO LAUF ANPASSEN ====================
RESULTS_DIR = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/KN2045_Mix/results/KN2045_Mix_3H_LK"
OUT_DIR = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/KN2045_Mix/h2_check_3H_LK"
YEARS = [2025, 2030, 2035, 2040, 2045, 2050]
PATTERN = "base_s_27__none_{year}.nc"

# Ordner, in dem h2_storage_check.py liegt
SCRIPT_DIR = "/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test/analysis/h2_storage/h2_storage_check.py"  # Ordner mit h2_storage_check.py

In [0]:
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if SCRIPT_DIR not in sys.path:
    sys.path.append(SCRIPT_DIR)
import h2_storage_check as hc

importlib.reload(hc)  # nach git pull ohne Kernel-Neustart aktuelle Version laden
HS = hc.HS_PER_HI

from IPython.display import HTML
from IPython.display import display as ipy_display


def show(df, fmt=None):
    # Tabelle als HTML anzeigen. Das display() von Databricks wandelt pandas in Spark um,
    # scheitert an gemischten Typen und verwirft den Zeilenindex.
    if fmt is not None:
        html = df.style.format(fmt, na_rep="–").to_html()
    else:
        html = df.to_html(na_rep="–")
    ipy_display(HTML(html))

## 1. Netze auswerten

In [0]:
summary, balances, extras = hc.run(
    RESULTS_DIR, years=YEARS, out_dir=OUT_DIR, pattern=PATTERN, return_extras=True
)

## 2. Phase-0-Checks

- `n_snapshots`: 8760 = stündlich, 2920 = 3H, 24 = 365h (dann Spitzen nicht belastbar)
- `balance_residual_max_abs_MW` sollte ≈ 0 sein, sonst fehlt eine Komponente in der Bilanz
- `store_charger_links` = 0 heißt: keine Leistungsbegrenzung oder -kosten für Ein-/Ausspeicherung

In [0]:
phase0 = [
    "n_snapshots", "weighting_unique_h", "sum_weights_h",
    "config_clustering_temporal", "config_snapshots", "config_scenario",
    "n_DE_H2_buses", "cavern_buses", "tank_buses",
    "cavern_e_nom_opt_TWh", "cavern_e_nom_max_TWh", "tank_e_nom_opt_TWh",
    "cavern_capital_cost_EUR_per_MWh_a", "tank_capital_cost_EUR_per_MWh_a",
    "store_charger_links", "balance_residual_max_abs_MW",
    "electrolysis_GWel", "h2_power_carriers", "h2_power_GWel", "h2_power_GW_H2in",
]
show(summary.loc[[k for k in phase0 if k in summary.index]])

## 3. Kennzahlen ESL / ASL je Jahr

In [0]:
def kpi_table(summary, prefix="all_"):
    s = summary.T
    t = pd.DataFrame(index=s.index)
    for lab in ["snap", "day"]:
        t[f"ASL max {lab} [GW_Hi]"] = s[f"{prefix}ASL_max_{lab}"].astype(float) / 1e3
        t[f"ESL max {lab} [GW_Hi]"] = s[f"{prefix}ESL_max_{lab}"].astype(float) / 1e3
        t[f"ESL/ASL max {lab}"] = s[f"{prefix}ESL/ASL_max_{lab}"].astype(float)
        t[f"ASL q99.5 {lab} [GW_Hi]"] = s[f"{prefix}ASL_q995_{lab}"].astype(float) / 1e3
        t[f"ESL q99.5 {lab} [GW_Hi]"] = s[f"{prefix}ESL_q995_{lab}"].astype(float) / 1e3
    t["ASL max [GW_Hs]"] = t["ASL max snap [GW_Hi]"] * HS
    t["ESL max [GW_Hs]"] = t["ESL max snap [GW_Hi]"] * HS
    t["Volllaststunden ASL [h]"] = s[f"{prefix}FLH_ASL_h"].astype(float)
    t["Volllaststunden ESL [h]"] = s[f"{prefix}FLH_ESL_h"].astype(float)
    t["Ausspeicherung [TWh_Hi]"] = s[f"{prefix}discharge_TWh"].astype(float)
    if prefix == "all_":
        t["AGV genutzt [TWh_Hi]"] = s["AGV_used_TWh"].astype(float)
        t["AGV genutzt [TWh_Hs]"] = t["AGV genutzt [TWh_Hi]"] * HS
        t["Zyklen"] = s["cycles"].astype(float)
    return t

kpi = kpi_table(summary)
fmt = {c: ("{:.0%}" if "ESL/ASL" in c else "{:.1f}") for c in kpi.columns}
show(kpi, fmt)

In [0]:
# 2037 (NEP-Stützjahr) linear aus 2035 und 2040
if {2035, 2040} <= set(kpi.index):
    k2037 = kpi.loc[2035] + (kpi.loc[2040] - kpi.loc[2035]) * 2 / 5
    k2037["ESL/ASL max snap"] = k2037["ESL max snap [GW_Hi]"] / k2037["ASL max snap [GW_Hi]"]
    k2037["ESL/ASL max day"] = k2037["ESL max day [GW_Hi]"] / k2037["ASL max day [GW_Hi]"]
    show(k2037.to_frame("2037 (interpoliert)").T, fmt)

## 4. Jahresbilanz der deutschen H2-Busse [TWh_Hi]

In [0]:
annual = summary.loc[[k for k in summary.index if k.startswith("annual_TWh")]].astype(float)
annual.index = annual.index.str.replace("annual_TWh[", "", regex=False).str.rstrip("]")
show(annual.round(1))

## 5. Dauerlinien Speicher-Dispatch (Deutschland, alle H2-Speicher)

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))
for y, inj in balances.items():
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1) / 1e3
    w = inj.index.to_series().diff().dt.total_seconds().div(3600).bfill()
    s = p.sort_values(ascending=False)
    hours = w.loc[s.index].cumsum()
    ax.plot(hours.values, s.values, label=str(y))
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Stunden im Jahr")
ax.set_ylabel("GW_Hi  (> 0 Ausspeicherung, < 0 Einspeicherung)")
ax.set_title("Dauerlinie H2-Speicher Deutschland")
ax.legend()
plt.show()

## 6. Zerlegung von R(t) in den Spitzenzeitpunkten

Beiträge aller Komponenten zur Bilanz der deutschen H2-Busse im Zeitschritt mit maximaler
Ausspeicherung (ASL) und maximaler Einspeicherung (ESL). Positiv = Einspeisung in den H2-Bus.

In [0]:
rows = {}
for y, inj in balances.items():
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1)
    for lab, t in {"ASL-Spitze": p.idxmax(), "ESL-Spitze": p.idxmin()}.items():
        r = inj.loc[t].drop("balance residual") / 1e3
        r.name = (y, lab, str(t))
        rows[r.name] = r
peaks = pd.DataFrame(rows).fillna(0.0)
peaks = peaks.loc[peaks.abs().max(axis=1) > 0.05]
show(peaks.round(1))

## 7. Zeitreihe eines Jahres

In [0]:
YEAR = 2045
inj = balances[YEAR] / 1e3
daily = inj.drop(columns="balance residual").resample("D").mean()
daily = daily.loc[:, daily.abs().max() > 0.1]
fig, ax = plt.subplots(figsize=(14, 6))
daily.plot(ax=ax, lw=1)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("GW_Hi (Tagesmittel, > 0 = Einspeisung in H2-Bus)")
ax.set_title(f"H2-Bilanz Deutschland {YEAR}")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

store_cols = [c for c in balances[YEAR].columns if c.startswith("H2 Store")]
w = balances[YEAR].index.to_series().diff().dt.total_seconds().div(3600).bfill()
soc = (-(balances[YEAR][store_cols].sum(axis=1) * w).cumsum()) / 1e6
fig, ax = plt.subplots(figsize=(14, 3))
(soc - soc.min()).plot(ax=ax)
ax.set_ylabel("TWh_Hi")
ax.set_title(f"Füllstand H2-Speicher Deutschland {YEAR} (relativ zum Minimum)")
plt.show()

## 8. Phase 1: Kennzahlen zur Herleitung von ESL und ASL

Verschiedene Definitionen der Leistung, weil die Netzplanung nicht unbedingt die
absolute Spitze eines einzelnen Zeitschritts braucht:
- **max**: Maximum je Zeitschritt
- **q99,5 / q99**: Quantile, d. h. die höchsten 0,5 % bzw. 1 % der Zeit werden ausgeblendet
- **roll24h / roll72h / roll168h / roll336h**: Maximum des gleitenden Mittels (wie lange wird die Leistung gehalten)

Dazu Befüllbarkeit: Tage, die das genutzte AGV mit der ESL zum Füllen braucht,
und die ESL nach 75-%-Regel des SR 2025 im Vergleich.

In [0]:
def phase1_table(summary):
    s = summary.T
    gw = lambda k: s[k].astype(float) / 1e3
    rows = {}
    for lab, a, e in [
        ("max", "all_ASL_max_snap", "all_ESL_max_snap"),
        ("q99,5", "all_ASL_q995_snap", "all_ESL_q995_snap"),
        ("q99", "all_ASL_q99_snap", "all_ESL_q99_snap"),
        ("Tagesmittel max", "all_ASL_max_day", "all_ESL_max_day"),
        ("roll72h", "all_ASL_max_roll72h", "all_ESL_max_roll72h"),
        ("roll168h", "all_ASL_max_roll168h", "all_ESL_max_roll168h"),
        ("roll336h", "all_ASL_max_roll336h", "all_ESL_max_roll336h"),
    ]:
        if a not in s:
            continue
        asl, esl = gw(a), gw(e)
        rows[f"ASL {lab} [GW_Hi]"] = asl
        rows[f"ESL {lab} [GW_Hi]"] = esl
        rows[f"ESL/ASL {lab}"] = (esl / asl).where(asl.abs() > 0.1)
    rows["ASL max [GW_Hs]"] = gw("all_ASL_max_snap") * HS
    rows["ESL max [GW_Hs]"] = gw("all_ESL_max_snap") * HS
    rows["AGV genutzt [TWh_Hi]"] = s["AGV_used_TWh"].astype(float)
    rows["Füllzeit mit ESL max [d]"] = s["fill_time_days_ESL_max_snap"].astype(float)
    rows["ESL nach 75-%-Regel [GW_Hi]"] = s["ESL_rule_75pct_GW"].astype(float)
    rows["Füllzeit mit 75-%-ESL [d]"] = s["fill_time_days_ESL_rule_75pct"].astype(float)
    rows["Import max [GW_Hi]"] = s.get("import_max_GW", pd.Series(np.nan, index=s.index)).astype(float)
    rows["Import-Pipelinekapazität [GW_Hi]"] = s["import_pipeline_capacity_GW"].astype(float)
    return pd.DataFrame(rows).T

p1 = phase1_table(summary)
def _fmt_row(row):
    pct = row.name.startswith("ESL/ASL")
    return row.map(lambda v: "–" if pd.isna(v) else (f"{v:.0%}" if pct else f"{v:.1f}"))

show(p1.apply(_fmt_row, axis=1))

## 9. Vergleich mit Referenzwerten (Briefing)

Werte in GW_Hi (SR 2025 aus GWh/h_Hs umgerechnet). Für 2037 wird das Modell aus 2035 und 2040 interpoliert.

In [0]:
REF = pd.DataFrame(
    [
        # Quelle, Jahr, ESL, ASL, AGV [TWh_Hi]
        ("MAHS 2026 oben", 2035, 5, 13, 7.6),
        ("MAHS 2026 oben", 2040, 9, 24, 17.5),
        ("MAHS 2026 oben", 2045, 18, 46, 28.3),
        ("MAHS 2026 unten", 2045, 12, 29, np.nan),
        ("MAHS 2024 oben", 2035, 27, 39, np.nan),
        ("MAHS 2024 oben", 2040, 30, 48, np.nan),
        ("MAHS 2024 oben", 2045, 41, 48, np.nan),
        ("ÜNB/FNB-Abfrage 2026", 2035, 16, 17, np.nan),
        ("ÜNB/FNB-Abfrage 2026", 2040, 38, 41, np.nan),
        ("ÜNB/FNB-Abfrage 2026", 2045, 55, 61, np.nan),
        ("EWI 2024 Start", 2045, 17, 106, np.nan),
        ("LFS O45-Strom (abgelesen)", 2045, 37.5, 160, np.nan),
        ("SR 2025 BNetzA", 2037, 26 / HS, 36 / HS, np.nan),
        ("SR 2025 BNetzA", 2045, 26 / HS, 36 / HS, np.nan),
    ],
    columns=["Quelle", "Jahr", "ESL [GW_Hi]", "ASL [GW_Hi]", "AGV [TWh_Hi]"],
)

model = pd.DataFrame({
    "Quelle": "PyPSA-DE (dieser Lauf)",
    "Jahr": p1.columns.astype(int),
    "ESL [GW_Hi]": p1.loc["ESL max [GW_Hi]"].values,
    "ASL [GW_Hi]": p1.loc["ASL max [GW_Hi]"].values,
    "AGV [TWh_Hi]": p1.loc["AGV genutzt [TWh_Hi]"].values,
})
if {2035, 2040} <= set(model.Jahr):
    m35, m40 = model.set_index("Jahr").loc[2035], model.set_index("Jahr").loc[2040]
    m37 = m35[["ESL [GW_Hi]", "ASL [GW_Hi]", "AGV [TWh_Hi]"]] * 0.6 + m40[["ESL [GW_Hi]", "ASL [GW_Hi]", "AGV [TWh_Hi]"]] * 0.4
    model = pd.concat([model, pd.DataFrame([{"Quelle": "PyPSA-DE (dieser Lauf)", "Jahr": 2037, **m37.to_dict()}])])

comp = pd.concat([model, REF]).reset_index(drop=True)
comp["ESL/ASL"] = comp["ESL [GW_Hi]"] / comp["ASL [GW_Hi]"]
comp = comp[comp.Jahr.isin([2035, 2037, 2040, 2045])].sort_values(["Jahr", "Quelle"])
show(comp.set_index(["Jahr", "Quelle"]),
     {"ESL [GW_Hi]": "{:.1f}", "ASL [GW_Hi]": "{:.1f}", "AGV [TWh_Hi]": "{:.1f}", "ESL/ASL": "{:.0%}"})

fig, ax = plt.subplots(figsize=(8, 6))
for q, g in comp.groupby("Quelle"):
    style = dict(marker="o", ms=9, zorder=3) if q.startswith("PyPSA") else dict(marker="s", alpha=0.7)
    ax.plot(g["ASL [GW_Hi]"], g["ESL [GW_Hi]"], ls="", label=q, **style)
    for _, r in g.iterrows():
        ax.annotate(int(r.Jahr), (r["ASL [GW_Hi]"], r["ESL [GW_Hi]"]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
lim = np.nanmax(comp[["ASL [GW_Hi]", "ESL [GW_Hi]"]].values) * 1.05
for r in [0.25, 0.5, 0.75, 1.0]:
    ax.plot([0, lim], [0, r * lim], color="grey", lw=0.6, ls=":")
    ax.text(lim, r * lim, f"{r:.0%}", fontsize=7, color="grey", va="bottom", ha="right")
ax.set_xlabel("ASL [GW_Hi]")
ax.set_ylabel("ESL [GW_Hi]")
ax.set_title("ESL vs. ASL: Modell und Referenzen")
ax.legend(fontsize=7, loc="upper left")
plt.show()

## 10. Regionale Sicht (deutsche Modellregionen)

Je H2-Bus: Speichertyp, AGV, ASL/ESL (nicht gleichzeitig!), Elektrolyse und H2-Kraftwerke.
Zeigt, wo Speicher sitzen und wo die Leistung gebraucht wird (Grundlage für Phase 4).

In [0]:
for y, ex in extras.items():
    print(f"=== {y} ===")
    show(ex["regional"].round(2))

## 11. H2-Speicher im Ausland

Prüft, ob ausländische Speicher (z. B. NL, DK) über die Importe deutsche Speicher ersetzen.
Spalten „Mittel in DE-Top-ASL/ESL“: Verhalten der Auslandsspeicher in den Zeitschritten,
in denen deutsche Speicher am stärksten aus- bzw. einspeichern.

In [0]:
for y, ex in extras.items():
    if ex["foreign"].empty:
        continue
    print(f"=== {y} ===")
    show(ex["foreign"].round(2))

## 12. Bericht für die Auswertung exportieren

Schreibt alle Tabellen in **eine** Textdatei `h2_report.txt` in `OUT_DIR`.
Diese Datei kann direkt an Claude gegeben werden (hochladen), statt Zellen einzeln zu kopieren.
Zusätzlich zu den Tabellen oben: Monatsbilanzen je Jahr und die mittlere Zusammensetzung
von R(t) in den Top-N-Zeitschritten mit höchster Aus- bzw. Einspeicherung.

In [0]:
import io
from pathlib import Path

TOP_N = 10  # Anzahl Spitzen-Zeitschritte für die Zerlegung

def _txt(df, floatfmt="{:.2f}"):
    with pd.option_context("display.max_rows", 500, "display.max_columns", 100, "display.width", 250):
        return df.to_string(float_format=lambda x: floatfmt.format(x))

buf = io.StringIO()
w = lambda *a: print(*a, file=buf)

w("# H2-Speicher ESL/ASL - Bericht")
w(f"RESULTS_DIR: {RESULTS_DIR}")
w(f"YEARS: {list(balances)}")
w("Einheiten: GW / TWh Heizwert (Hi), sofern nicht anders angegeben. Speicher > 0 = Ausspeicherung.")

w(); w("## Phase 0")
w(_txt(summary.loc[[k for k in phase0 if k in summary.index]]))

w(); w("## Kennzahlen je Jahr")
w(_txt(kpi.T))
if {2035, 2040} <= set(kpi.index):
    w(); w("## 2037 interpoliert")
    w(_txt(k2037.to_frame("2037")))

w(); w("## Phase 1: Kennzahlen (max, Quantile, gleitende Mittel, Befüllbarkeit)")
w(_txt(p1))
w(); w("## Vergleich mit Referenzen [GW_Hi, TWh_Hi]")
w(_txt(comp.set_index(["Jahr", "Quelle"]), "{:.2f}"))

w(); w("## Jahresbilanz deutsche H2-Busse [TWh]")
w(_txt(annual, "{:.1f}"))

w(); w("## Zerlegung im Zeitschritt der ASL- und ESL-Spitze [GW]")
w(_txt(peaks, "{:.1f}"))

for y, inj in balances.items():
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1)
    comp = inj.drop(columns="balance residual") / 1e3
    comp = comp.loc[:, comp.abs().max() > 0.05]
    wts = inj.index.to_series().diff().dt.total_seconds().div(3600).bfill()

    w(); w(f"## {y}: Monatsbilanz [TWh]")
    monthly = comp.mul(wts, axis=0).resample("MS").sum().div(1e3)
    monthly.index = monthly.index.month
    w(_txt(monthly, "{:.2f}"))

    for lab, idx in {"ASL": p.nlargest(TOP_N).index, "ESL": p.nsmallest(TOP_N).index}.items():
        w(); w(f"## {y}: Mittel der Top-{TOP_N} {lab}-Zeitschritte [GW]")
        w("Zeitpunkte: " + ", ".join(t.strftime("%m-%d %H") for t in sorted(idx)))
        w(_txt(comp.loc[idx].mean().to_frame("mean").T, "{:.1f}"))

    w(); w(f"## {y}: Regionen")
    w(_txt(extras[y]["regional"]))
    if not extras[y]["foreign"].empty:
        w(); w(f"## {y}: H2-Speicher im Ausland")
        w(_txt(extras[y]["foreign"]))

    daily = p.resample("D").mean() / 1e3
    w(); w(f"## {y}: Dauerlinie Speicher [GW], Quantile Zeitschritt und Tagesmittel")
    q = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999, 1]
    w(_txt(pd.DataFrame({"snap": (p / 1e3).quantile(q), "day": daily.quantile(q)}), "{:.1f}"))

report = buf.getvalue()
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUT_DIR) / "h2_report.txt").write_text(report, encoding="utf-8")
print(f"geschrieben: {OUT_DIR}/h2_report.txt ({len(report) / 1e3:.0f} kB)")

## Ergänzung (händsich)

In [0]:
# ===== Folie: Dauerlinien ASL vs. ESL (gleiche Achse) mit 75-%-Regel =====
YEARS_DL = [2040, 2045]
X_MAX_H = 3000
COLORS = {2035: "#7f7f7f", 2040: "#1f77b4", 2045: "#c00000", 2050: "#2e7d32"}

fig, ax = plt.subplots(figsize=(12, 5))
rows = {}
for y in YEARS_DL:
    inj = balances[y]
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1) / 1e3                                   # GW
    w = inj.index.to_series().diff().dt.total_seconds().div(3600).bfill()
    c = COLORS.get(y, None)
    for lab, s, ls in [("ASL", p.clip(lower=0), "-"), ("ESL", (-p).clip(lower=0), "--")]:
        s = s.sort_values(ascending=False)
        hours = w.loc[s.index].cumsum().values
        ax.plot(hours, s.values, ls=ls, color=c, lw=1.8, label=f"{lab} {y}")
    asl_max, esl_max = p.max(), -p.min()
    ax.axhline(0.75 * asl_max, color=c, lw=1, ls=":")
    ax.text(X_MAX_H * 0.99, 0.75 * asl_max, f"75-%-Regel {y}: ESL {0.75 * asl_max:.0f} GW",
            color=c, fontsize=8, ha="right", va="bottom")
    rows[y] = {
        "ASL max [GW_Hi]": asl_max,
        "ESL max [GW_Hi]": esl_max,
        "ESL/ASL": esl_max / asl_max,
        "ESL nach 75-%-Regel [GW_Hi]": 0.75 * asl_max,
        "Stunden ASL > 50 % ASL max": w[p > 0.5 * asl_max].sum(),
        "Stunden ESL > 50 % ESL max": w[-p > 0.5 * esl_max].sum(),
        "Stunden ESL > 75-%-Regel": w[-p > 0.75 * asl_max].sum(),
    }
ax.set_xlim(0, X_MAX_H)
ax.set_ylim(0, None)
ax.set_xlabel("Stunden im Jahr (absteigend sortiert)")
ax.set_ylabel("Leistung [GW_Hi]")
ax.set_title("Dauerlinien Aus- (ASL) und Einspeicherung (ESL), H2-Speicher Deutschland")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "folie_dauerlinie_asl_esl.png", dpi=200, bbox_inches="tight")
plt.show()

dl = pd.DataFrame(rows)
show(dl.apply(lambda r: r.map(lambda v: f"{v:.0%}" if r.name == "ESL/ASL" else f"{v:.0f}"), axis=1))